# Tutorial 50: `client.run_code` — the values API without the magic

The same round-trip as tutorial 14, but through the raw cas-client primitive
the `%%krauncher` magic is built on: a code *string* goes to the GPU task,
named local values ride in as `inputs=`, and variables computed remotely come
back via `outputs=[...]` through the relay's encrypted result mailbox — the
broker never stores task data.

The transfer set is **auto-detected** with `analyze_names` — the same AST
analysis the magic uses: free variables of the block are its inputs, assigned
names are its outputs. `lenient_outputs=True` makes the task return only the
values that are set and JSON-safe (the model, tensors etc. are dropped
remotely instead of failing the task).

Uses notebook top-level `await` (ipykernel runs the coroutine on its own
loop) — no IPython magic involved.


In [ ]:
%env CAS_CLIENT_CONFIG=../../cas-client/.env

In [ ]:
from krauncher import KrauncherClient
from krauncher.values import decode_outputs

client = KrauncherClient()

# The code block runs remotely as-is: `epochs`, `batch_size` and `val_images`
# arrive from inputs=...; `losses`, `val_preds` and `device_name` are picked
# up from its namespace by outputs=[...].
TRAIN_CODE = """
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        return self.pool(self.act(self.bn(self.conv(x))))

class ImageClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            ConvBlock(256, 512),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageClassifier(num_classes=10).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
losses = []
for epoch in range(1, epochs + 1):
    epoch_loss = 0.0
    n_batches = 50
    for i in range(n_batches):
        images = torch.randn(batch_size, 3, 64, 64, device=device)
        labels = torch.randint(0, 10, (batch_size,), device=device)

        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()

    avg = epoch_loss / n_batches
    losses.append(round(avg, 4))
    print(f"Epoch {epoch}/{epochs}  avg_loss={avg:.4f}")

# Classify the validation batch shipped from the client's namespace.
model.eval()
with torch.no_grad():
    val = torch.tensor(val_images, dtype=torch.float32, device=device)
    val_preds = model(val).argmax(dim=1).tolist()
device_name = str(device)
"""

In [ ]:
import random

from krauncher.codeblock import analyze_names

# A small validation batch generated locally: 8 images, 3x32x32.
# Plain nested lists — JSON-safe, well inside the 16 MB inline budget.
val_images = [
    [[[random.random() for _ in range(32)] for _ in range(32)] for _ in range(3)]
    for _ in range(8)
]

# Auto-detect the transfer set from the code block itself.
free, assigned = analyze_names(TRAIN_CODE)
print(f"inputs (free variables): {free}")
print(f"outputs (assigned names): {assigned}")

local_values = {"epochs": 3, "batch_size": 32, "val_images": val_images}
handle = await client.run_code(
    TRAIN_CODE,
    inputs={n: local_values[n] for n in free},
    outputs=assigned,
    lenient_outputs=True,  # drop unset / non-JSON-safe names remotely
    pip=["torch"],
    timeout=300,
)
print(f"Task ID: {handle.task_id}")
c = handle.classification
print(f"Classification: tier={c.tier}, vram={c.min_vram_gb}GB, CU={c.compute_units}, method={c.analysis_method}")

result = await handle.wait()

# Only the transferable subset of `assigned` comes back — pick what we need.
values = result.output
print(f"returned: {sorted(values)}")
print(f"Losses per epoch: {values['losses']}")
print(f"Validation predictions: {values['val_preds']}")
print(f"Trained on: {values['device_name']}")
print(f"Worker: {result.worker_id}  GPU: {result.actual_gpu}  Time: {result.execution_time_sec:.2f}s")
